# Replication of vanilla RNN with numpy.

### Model is created for prediction of the next element of the sequence based on previous element

At time step $t$, the vanilla RNN updates are:

$$
h_t = \tanh(x_t \cdot W_{xh} + W_{hh} \cdot h_{t-1} + b_h)
$$

$$
y_t = W_{hy} \cdot h_t + b_y
$$


In [3]:
import torch
import torch.nn.init as init 

In [74]:
# Creating the class with forward and backward propagations
class RNN():
    def __init__(self):
        # Initializing weights and biases with xavier uniform
        self.W_xh = torch.empty(1, 50).requires_grad_(True)
        self.W_hh = torch.empty(50, 50).requires_grad_(True)
        self.W_hy = torch.empty(1, 50).requires_grad_(True)
        init.xavier_uniform_(self.W_xh)
        init.xavier_uniform_(self.W_hh)
        init.xavier_uniform_(self.W_hy)
        self.b_h = torch.zeros(50).requires_grad_(True)
        self.b_y = torch.zeros(1).requires_grad_(True)

        # Making the default initial hidden state
        self.h_init = torch.ones((50,), requires_grad=True)  # WARNING - why shape is not (5,5)?
    
    # Uses "n" element to predict "n+1" element
    def forward(self, inputs, targets):
        # Check the x and y lengths
        assert len(inputs)==len(targets)
        
        # The MSE loss
        self.loss = 0
        # Counter of accurate predictions
        acc_counter = []
        # Making the initialized hidden state as prior h_t 
        h_t = self.h_init
        for i in range(len(inputs)):
            x_input = inputs[i].unsqueeze(0) # WARNING - to make it as 1d, form factor
            h_t = torch.tanh(x_input@self.W_xh + self.W_hh@h_t + self.b_h)
            y_t = self.W_hy@h_t+self.b_y
            if i!=len(inputs)-1:
                y_real = targets[i]
                difference = abs((float(y_t)*2048)-(float(y_real)*2048))
                isCorrect = difference<0.1
                print(f"Predicted: {(float(y_t)*2048)} Real: {(float(y_real)*2048)} IsCorrect: {isCorrect} Difference {difference}")
                acc_counter.append(isCorrect)
                # Calculating the loss for this timestep
                loss_t = (y_real-y_t)**2
                self.loss+=loss_t
            else:
                self.loss = self.loss / (len(inputs)) # WARNING - why it is better to divide the loss after?
                return y_t, sum(acc_counter)
    
    # Backpropagation + updating the weights, biases + zeroing the gradients for future
    def backward_and_update(self, lr=0.1):
        self.loss.backward()
        
        # Clipping the gradients so the minimum of the loss function wouldn't be accidentally jumped over
        torch.nn.utils.clip_grad_norm_([self.W_xh, self.W_hh, self.W_hy, self.b_h, self.b_y], max_norm=1.0)
        
        try:
            self.W_xh.data -= lr*self.W_xh.grad
            self.W_hh.data -= lr*self.W_hh.grad
            self.b_h.data -= lr*self.b_h.grad
            self.W_hy.data -= lr*self.W_hy.grad
            self.b_y.data -= lr*self.b_y.grad
            
            # Manually zero gradients
            self.W_xh.grad.zero_()
            self.W_hh.grad.zero_()
            self.b_h.grad.zero_()
            self.W_hy.grad.zero_()
            self.b_y.grad.zero_()
        except TypeError as e:
            print("Caught TypeError:", e)
            print("Developer: probably because there was no forward() called")
        

# WARNING - how vanishing/exploding of gradients happen in here?

# WARNING - the model somehow don't converge with non normalized inputs
inputs = torch.tensor([1,2,4,8,16,32,64,128,256,512,1024,2048], dtype=torch.float32)/2048
outputs = inputs*2
model = RNN()
for i in range(5000):
    random_indices = torch.randperm(len(inputs))  # Randomly shuffles indices [0, 1, 2, ..., n-1]
    reordered_inputs = inputs[random_indices]
    reordered_outputs = outputs[random_indices]
    prediction, accuracy = model.forward(reordered_inputs, reordered_outputs)
    model.backward_and_update()

Predicted: -259.553466796875 Real: 512.0 IsCorrect: False Difference 771.553466796875
Predicted: 1119.1240234375 Real: 2.0 IsCorrect: False Difference 1117.1240234375
Predicted: 592.8641357421875 Real: 64.0 IsCorrect: False Difference 528.8641357421875
Predicted: 638.3095092773438 Real: 1024.0 IsCorrect: False Difference 385.69049072265625
Predicted: 1511.6248779296875 Real: 128.0 IsCorrect: False Difference 1383.6248779296875
Predicted: -91.52764892578125 Real: 4.0 IsCorrect: False Difference 95.52764892578125
Predicted: 1456.51513671875 Real: 8.0 IsCorrect: False Difference 1448.51513671875
Predicted: 1277.078125 Real: 16.0 IsCorrect: False Difference 1261.078125
Predicted: 277.0994567871094 Real: 4096.0 IsCorrect: False Difference 3818.9005432128906
Predicted: 2082.042236328125 Real: 2048.0 IsCorrect: False Difference 34.042236328125
Predicted: 971.8555297851562 Real: 256.0 IsCorrect: False Difference 715.8555297851562
Predicted: 140.4976806640625 Real: 2048.0 IsCorrect: False Diffe

In [69]:
torch.tensor([1,2,4,8,16,32,64,128,256,512,1024,2048], dtype=torch.float32)/2048
model.forward(inputs, outputs)

Predicted: 2.0005722045898438 Real: 2.0 IsCorrect: True
Predicted: 4.000297546386719 Real: 4.0 IsCorrect: True
Predicted: 8.000205993652344 Real: 8.0 IsCorrect: True
Predicted: 16.00023651123047 Real: 16.0 IsCorrect: True
Predicted: 31.99376678466797 Real: 32.0 IsCorrect: True
Predicted: 64.0049057006836 Real: 64.0 IsCorrect: True
Predicted: 127.98656463623047 Real: 128.0 IsCorrect: True
Predicted: 256.014892578125 Real: 256.0 IsCorrect: True
Predicted: 511.990966796875 Real: 512.0 IsCorrect: True
Predicted: 1024.0155029296875 Real: 1024.0 IsCorrect: True
Predicted: 2047.989990234375 Real: 2048.0 IsCorrect: True


(tensor([0.5007], grad_fn=<AddBackward0>), 11)

# Check by using the next number in the exponentially growing sequence

In [ ]:
# TODO: check...

In [43]:
test_sequence = torch.tensor([2, 4, 8])/2048
prediction, accuracy = model.forward(long_sequence)
prediction*2048

Predicted: 1.9999217987060547 Real: 2.0 IsCorrect: True
Predicted: 3.9998607635498047 Real: 4.0 IsCorrect: True
Predicted: 8.000532150268555 Real: 8.0 IsCorrect: True
Predicted: 15.999067306518555 Real: 16.0 IsCorrect: True
Predicted: 32.00065612792969 Real: 32.0 IsCorrect: True
Predicted: 63.99967956542969 Real: 64.0 IsCorrect: True
Predicted: 127.99992370605469 Real: 128.0 IsCorrect: True
Predicted: 256.00067138671875 Real: 256.0 IsCorrect: True
Predicted: 511.99908447265625 Real: 512.0 IsCorrect: True
Predicted: 1024.0008544921875 Real: 1024.0 IsCorrect: True
Predicted: 2047.9996337890625 Real: 2048.0 IsCorrect: True


tensor([2011.1375], grad_fn=<MulBackward0>)

In [49]:
test_sequence = torch.tensor([1,2])/2048
prediction, accuracy = model.forward(test_sequence)
prediction*2048

Predicted: 2.000295639038086 Real: 2.0 IsCorrect: True


tensor([3.9998], grad_fn=<MulBackward0>)